# Synthetic Adaptive Conformal Experiments

Helpers live in `adaptive_conformal.py`.

Contents:

1. Data: two-stage causal process `w → x → y` with two *relationship* shifts
   (the w→x slope, then the x→y slope, with a pause in the middle).
2. Six adaptive methods run on the residual scores: our adaptive Split Residuals (also sometimes called FWER in code), ACI
   (clipped), PID with scorecaster, decaying quantile (OCID), ECI, and DtACI.
3-4. Model-shift figures: prediction-interval widths and 200-step sliding
   coverage.
5. Covariate-shift variant: the comparison when `w`'s *distribution*
   shifts instead of slope.
6. Adaptivity to shift on internal weights: a/b weights, residual component score ratios, no-c/d adjustment ablation.
7. Four adaptive parameter sweeps (gamma=`alpha_lr`, eta=`lr`, delta, window)


In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from adaptive_conformal import (
    sliding_window_average,
    adaptive_fwer_alpha, aci_clipped, quantile_integrator_log_scorecaster,
    ECI, decay_quantile, dtaci,
)

## 1. Data and fitted models

Generate `(w, x, y)` of length 2100 with:
- baseline: `x = 3w + ε`, `y = 4x + ε`
- upstream covariate shift in `[size-1200, size-800]`: `x` regressed on `w` with stronger slope + mean shift
- downstream relationship shift in `[size-400, size]`: `y = 7x + 5 + ε`

Fit two ridge OLS models on the first 500 samples; compute the three residual scores needed for all methods

In [ ]:
np.random.seed(42)

epsilon = 0.05
alpha = 0.1
delta = 0.5
window_length = 100
lr = 0.01
alpha_lr = 0.01
ahead = 1
cal_size = 3 * window_length // 4
T_burnin = window_length

grid = np.linspace(0.0, 1, 12)
la, lb = np.meshgrid(grid, grid)
lambda_set = np.column_stack((la.flatten(), lb.flatten()))

size = 2100
train_size = 500
shift_location_1 = 1200      # upstream shift window length
gap = 400
shift_location_2 = 400       # downstream shift window length (last segment)

w = np.random.normal(0, 1, size)
x = w * 3 + np.random.normal(0, 1, size)
x[-shift_location_1:-shift_location_2 - gap] = (
    w[-shift_location_1:-shift_location_2 - gap] * 8
    + np.random.normal(1, 1, shift_location_1 - shift_location_2 - gap)
)
y = x * 4 + np.random.normal(0, 1, size)
y[-shift_location_2:] = x[-shift_location_2:] * 7 + np.random.normal(5, 1, shift_location_2)

full_data = np.vstack([w, x, y])
train = full_data[:, :train_size]
not_train = full_data[:, train_size:]

reg = 0.1
res1 = sm.OLS(train.T[:, 1], train.T[:, 0]).fit_regularized(alpha=reg, L1_wt=0)
res2 = sm.OLS(train.T[:, 2], train.T[:, 1]).fit_regularized(alpha=reg, L1_wt=0)

# Three residual score streams used downstream.
scores = np.abs(not_train[-1, :] - res2.predict(res1.predict(not_train[0, :])))
r2_scores = np.abs(not_train[-1, :] - res2.predict(not_train[1, :]))
r1_scores = np.abs(r2_scores - scores)

## 2. Run the six adaptive methods

Only methods plotted in cells 4–5 are computed.

In [ ]:
res = adaptive_fwer_alpha(
    scores, alpha, epsilon, window_length, cal_size, T_burnin,
    lambda_set, r1_scores, r2_scores, delta,
    None, ahead, lr, alpha_lr, tol=0.0,
)

res_aci = aci_clipped(scores, alpha, alpha_lr, window_length, T_burnin, ahead)

Csat = 2 / np.pi * (np.ceil(np.log(size - T_burnin) * 0.1) - 1 / np.log(size - T_burnin))
KI = 20
res_pid_sc = quantile_integrator_log_scorecaster(
    scores, alpha, alpha_lr, not_train, T_burnin, Csat, KI, 'upper', ahead,
    integrate=True, proportional_lr=True, scorecast=True,
)

res_eci = ECI(scores, alpha, 0.01, T_burnin, ahead)

# Decaying quantile = the OCID implementation 
eps_d = 0.1
etas_decaying = np.array([40 / (t ** (1 / 2 + eps_d)) for t in range(1, len(scores) + 1)])
res_decay = decay_quantile(scores, scores[0], etas_decaying, alpha, ahead)

steps = [0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.128]
res_dtaci = dtaci(scores, alpha, steps, ahead, window_length)

## 3. Figure — prediction-interval widths (model shift)

In [ ]:
window_end = -shift_location_1 - 100

series = [
    (res['q'][window_end:],          'Split Residuals', '-',  'C0'),
    (res_aci['q'][window_end:],      'ACI (clipped)',   '--', 'C1'),
    (res_pid_sc['q'][window_end:],   'PID',             '-.', 'C2'),
    (res_decay['q'][window_end:],    'OCID',            '-.', 'C3'),
    (res_eci['q'][window_end:],      'ECI',             '--', 'C4'),
    (res_dtaci['q'][window_end:],    'DtACI',           '-.', 'C5'),
]
n_post = len(series[0][0])

for arr, label, ls, color in series:
    plt.plot(arr, label=label, linestyle=ls, color=color)

plt.axvline(100,                                 color='gray',      linestyle=':', linewidth=2, label='Shock (Upstream)')
plt.axvline(n_post - shift_location_2 - gap,     color='slategray', linestyle=':', linewidth=2, label='Return')
plt.axvline(n_post - shift_location_2,           color='black',     linestyle=':', linewidth=2, label='Shock (Downstream)')

plt.xlabel('Timestep')
plt.ylabel('Prediction Interval Width')
plt.legend(loc='lower right', frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig('pi_synthetic_widths_plot_model_shift.png', bbox_inches='tight', dpi=300)
plt.show()

## 4. Figure — 200-step sliding coverage (model shift)

In [ ]:
cov_series = [
    (res['cov'],         'Split Residuals', '-',  'C0'),
    (res_aci['cov'],     'ACI (clipped)',   '--', 'C1'),
    (res_pid_sc['cov'],  'PID',             '-.', 'C2'),
    (res_decay['cov'],   'OCID',            '--', 'C3'),
    (res_eci['cov'],     'ECI',             '--', 'C4'),
    (res_dtaci['cov'],   'DtACI',           '-.', 'C5'),
]

for cov, label, ls, color in cov_series:
    arr = sliding_window_average(cov, 200)
    plt.plot(arr[window_end:], label=label, linestyle=ls, color=color)

plt.axhline(0.9, color='gray', linestyle='--', linewidth=1, label='90% Coverage')
n_post = len(sliding_window_average(cov_series[0][0], 200)[window_end:])
plt.axvline(100,                             color='gray',      linestyle=':', linewidth=2, label='Shock (Upstream)')
plt.axvline(n_post - shift_location_2 - gap, color='slategray', linestyle=':', linewidth=2, label='Return')
plt.axvline(n_post - shift_location_2,       color='black',     linestyle=':', linewidth=2, label='Shock (Downstream)')

plt.xlabel('Timestep')
plt.ylabel('Average Coverage (200 Sliding Window)')
plt.ylim(0.6, 1.0)
plt.legend(loc='lower right', frameon=False, fontsize=7)
plt.savefig('pi_synthetic_cov_plot_model_shift.png', bbox_inches='tight', dpi=300)
plt.show()

## 5. Covariate-shift variant

Same three-method comparison, but now the *covariate* `w` shifts distribution while the relationships stay
fixed.

In [ ]:
np.random.seed(42)
delta_cov = 0.3

w_cov = np.random.normal(0, 1, size)
w_cov[-shift_location_1:-shift_location_2 - gap] = np.random.normal(3, 2, shift_location_1 - shift_location_2 - gap)
w_cov[-shift_location_2:] = np.random.normal(-3, 2, shift_location_2)
x_cov = w_cov * 3 + np.random.normal(0, 1, size)
y_cov = x_cov * 4 + np.random.normal(0, 1, size)

full_cov = np.vstack([w_cov, x_cov, y_cov])
train_cov = full_cov[:, :train_size]
not_train_cov = full_cov[:, train_size:]

res1_cov = sm.OLS(train_cov.T[:, 1], train_cov.T[:, 0]).fit_regularized(alpha=reg, L1_wt=0)
res2_cov = sm.OLS(train_cov.T[:, 2], train_cov.T[:, 1]).fit_regularized(alpha=reg, L1_wt=0)

scores_cov = np.abs(not_train_cov[-1, :] - res2_cov.predict(res1_cov.predict(not_train_cov[0, :])))
r2_scores_cov = np.abs(not_train_cov[-1, :] - res2_cov.predict(not_train_cov[1, :]))
r1_scores_cov = np.abs(r2_scores_cov - scores_cov)

res_cov = adaptive_fwer_alpha(
    scores_cov, alpha, epsilon, window_length, cal_size, T_burnin,
    lambda_set, r1_scores_cov, r2_scores_cov, delta_cov,
    None, ahead, lr, alpha_lr,
)
res_aci_cov = aci_clipped(scores_cov, alpha, alpha_lr, window_length, T_burnin, ahead)
Csat = 2 / np.pi * (np.ceil(np.log(size - T_burnin) * 0.1) - 1 / np.log(size - T_burnin))
KI = 20
res_pid_sc_cov = quantile_integrator_log_scorecaster(
    scores_cov, alpha, alpha_lr, not_train_cov, T_burnin, Csat, KI, 'upper', ahead,
    integrate=True, proportional_lr=True, scorecast=True,
)

### Fig — widths (covariate shift)

In [ ]:
n_post = shift_location_1 + 100
above_us = res_cov['q'][-n_post:]
above_aci = res_aci_cov['q'][-n_post:]
above_pid_sc = res_pid_sc_cov['q'][-n_post:]
plt.plot(above_us, label='Split Residuals', linestyle='-', color='C0')
plt.plot(above_aci, label='ACI (clipped)', linestyle='--', color='C1')
plt.plot(above_pid_sc, label='PID', linestyle='-.', color='C2')
plt.xlabel('Timestep')
plt.axvline(100, color='gray', linestyle=':', linewidth=2, label='Shock (Upstream)')
plt.axvline(n_post - shift_location_2 - gap, color='slategray', linestyle=':', linewidth=2, label='Return')
plt.axvline(n_post - shift_location_2, color='black', linestyle=':', linewidth=2, label='Shock (Downstream)')
plt.ylabel('Prediction Interval Width')
plt.legend(loc='lower right', frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig('pi_synthetic_widths_plot_covariate.png', bbox_inches='tight', dpi=300)
plt.show()

### Fig — coverage (covariate shift)

In [ ]:
n_post = shift_location_1 + 100
for cov, win, label, ls, color in [
    (res_cov['cov'],        200, 'Split Residuals', '-',  'C0'),
    (res_aci_cov['cov'],    200, 'ACI (clipped)',   '--', 'C1'),
    (res_pid_sc_cov['cov'], 200, 'PID',             '-.', 'C2'),
]:
    plt.plot(sliding_window_average(cov, win)[-n_post:], linestyle=ls, color=color, label=label)
plt.axhline(0.9, color='gray', linestyle='--', linewidth=1, label='90% Coverage')
plt.xlabel('Timestep')
plt.axvline(100, color='gray', linestyle=':', linewidth=2, label='Shock (Upstream)')
plt.axvline(n_post - shift_location_2 - gap, color='slategray', linestyle=':', linewidth=2, label='Return')
plt.axvline(n_post - shift_location_2, color='black', linestyle=':', linewidth=2, label='Shock (Downstream)')
plt.ylabel('Average Coverage (200 Sliding Window)')
plt.ylim(0.6, 1.0)
plt.legend(loc='lower right', frameon=False, fontsize=7)
plt.savefig('pi_synthetic_cov_plot_covariate.png', bbox_inches='tight', dpi=300)
plt.show()

## 6. Split Residual internals under shift

Diagnostics for *how* the adaptive method behaves well across the shift
window, reusing the same `scores` / `r1_scores` / `r2_scores`

In [ ]:
# FWER internals use delta=0.3
delta_internals = 0.3

res_internals = adaptive_fwer_alpha(
    scores, alpha, epsilon, window_length, cal_size, T_burnin,
    lambda_set, r1_scores, r2_scores, delta_internals,
    None, ahead, lr, alpha_lr,
)

# 'No c/d adaptation' ablation: freeze the c/d learning rate (lr=0).
res_no_cd = adaptive_fwer_alpha(
    scores, alpha, epsilon, window_length, cal_size, T_burnin,
    lambda_set, r1_scores, r2_scores, delta_internals,
    None, ahead, 0, alpha_lr,
)

### Fig — learned `a`, `b` scaling weights

In [ ]:
plt.plot(res_internals['a'][-shift_location_1 - gap + 300:], color='firebrick', label=r'$a$ parameter')
plt.plot(res_internals['b'][-shift_location_1 - gap + 300:], color='rebeccapurple', label=r'$b$ parameter')
plt.xlabel('Timestep')
plt.ylabel('Scaling Value')
plt.axvline(100, color='black', linestyle=':', linewidth=2)
plt.axvline(500, color='black', linestyle=':', linewidth=2)
plt.axvline(900, color='black', linestyle=':', linewidth=2)
plt.legend(loc='lower right', frameon=False, fontsize=8)
plt.savefig('abplot.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig — residual component score ratios 

In [ ]:
sl = slice(-shift_location_1 - gap + 300, None)
plt.plot(np.divide(scores[sl], res_internals['r1'][sl]), color='firebrick', label=r'$R/\Delta R_1$')
plt.plot(np.divide(scores[sl], res_internals['r2'][sl]), color='rebeccapurple', label=r'$R/R_2$')
plt.xlabel('Timestep')
plt.ylabel('Ratio')
plt.legend(loc='upper right', frameon=False, fontsize=10)
plt.savefig('r2r1ratio.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig — `R₂` component width

In [ ]:
plt.plot(res_internals['r2'][-shift_location_1 - gap + 300:], color='rebeccapurple', label=r'$R_2$')
plt.xlabel('Timestep')
plt.ylabel('Width')
plt.legend(loc='lower right', frameon=False, fontsize=10)
plt.savefig('r2plot.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig — no-c/d ablation: coverage and width

In [ ]:
array_cov_no_cd = sliding_window_average(res_no_cd['cov'], 200)
plt.plot(array_cov_no_cd[-shift_location_1 - 100:], linestyle='-', color='C0', label='Split Residuals (no c,d)')
plt.axhline(0.9, color='gray', linestyle='--', linewidth=1, label='Minimum Coverage Level (90%)')
plt.xlabel('Timestep')
plt.ylabel('Average Coverage (200 Sliding Window)')
plt.ylim(0.6, 1.0)
plt.legend(loc='lower right', frameon=False, fontsize=10)
plt.savefig('nocd.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
above_us_no_cd = res_no_cd['q'][-shift_location_1 - 100:]
plt.plot(above_us_no_cd[-shift_location_1 - gap - shift_location_2:], label='Split Residuals', linestyle='-', color='C0')
plt.xlabel('Timestep')
plt.axvline(100, color='black', linestyle=':', linewidth=2)
plt.axvline(500, color='black', linestyle=':', linewidth=2)
plt.axvline(900, color='black', linestyle=':', linewidth=2)
plt.ylabel('Prediction Interval Width')
plt.legend(loc='upper right', frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig('nocdwidth.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Adaptive parameter sweeps

Sweep the four knobs of the adaptive method on the model-shift data, 50 reps
per value, comparing our adaptive FWER against ACI, PID, and OCID. For each
value we record interval width, minimum sliding coverage, abstention count,
the largest coverage gap vs. each baseline (and where it occurs), and the
recovery time after each shock. 


In [ ]:
def first_index_greater_than(arr, lower, upper, threshold=0.9):
    if lower < 0 or upper > len(arr) or lower >= upper:
        return np.nan
    idx = np.where(arr[lower:upper] > threshold)[0]
    return (idx[0] + lower) if idx.size > 0 else np.nan

# Order of metric rows saved to .npy. The gamma sweep saves all 23; the others
# save the first 11 but reorder the us_arg_* block (decay, aci, pid) to match
# the original notebook's layout.
_GAMMA_ROWS = [
    'us_width', 'us_min_cov', 'us_abstention',
    'us_max_vs_aci', 'us_max_vs_pid', 'us_max_vs_decay',
    'us_arg_aci', 'us_arg_pid', 'us_arg_decay',
    'us_recovery_up', 'us_recovery_down',
    'aci_width', 'aci_min_cov', 'aci_recovery_up', 'aci_recovery_down',
    'pid_width', 'pid_min_cov', 'pid_recovery_up', 'pid_recovery_down',
    'decay_width', 'decay_min_cov', 'decay_recovery_up', 'decay_recovery_down',
]
_US_ROWS = [
    'us_width', 'us_min_cov', 'us_abstention',
    'us_max_vs_aci', 'us_max_vs_pid', 'us_max_vs_decay',
    'us_arg_decay', 'us_arg_aci', 'us_arg_pid',
    'us_recovery_up', 'us_recovery_down',
]

def _adaptive_sweep_rep(*, alpha, epsilon, delta, window_length, cal_size, T_burnin,
                        lr, alpha_lr, ahead, lambda_set, size, train_size,
                        shift_location_1, gap, shift_location_2, reg):
    """One sweep rep on model-shift data; returns (metrics, plot_arrays)."""
    w = np.random.normal(0, 1, size)
    x = w * 3 + np.random.normal(0, 1, size)
    x[-shift_location_1:-shift_location_2 - gap] = (
        w[-shift_location_1:-shift_location_2 - gap] * 8
        + np.random.normal(1, 1, shift_location_1 - shift_location_2 - gap)
    )
    y = x * 4 + np.random.normal(0, 1, size)
    y[-shift_location_2:] = x[-shift_location_2:] * 7 + np.random.normal(5, 1, shift_location_2)

    full = np.vstack([w, x, y])
    train = full[:, :train_size]
    not_train = full[:, train_size:]
    res1 = sm.OLS(train.T[:, 1], train.T[:, 0]).fit_regularized(alpha=reg, L1_wt=0)
    res2 = sm.OLS(train.T[:, 2], train.T[:, 1]).fit_regularized(alpha=reg, L1_wt=0)
    scores = np.abs(not_train[-1, :] - res2.predict(res1.predict(not_train[0, :])))
    r2_scores = np.abs(not_train[-1, :] - res2.predict(not_train[1, :]))
    r1_scores = np.abs(r2_scores - scores)

    res = adaptive_fwer_alpha(
        scores, alpha, epsilon, window_length, cal_size, T_burnin,
        lambda_set, r1_scores, r2_scores, delta, None, ahead, lr, alpha_lr,
    )
    res_aci = aci_clipped(scores, alpha, alpha_lr, window_length, T_burnin, ahead)
    Csat = 2 / np.pi * (np.ceil(np.log(size - T_burnin) * 0.1) - 1 / np.log(size - T_burnin))
    KI = 20
    res_pid_sc = quantile_integrator_log_scorecaster(
        scores, alpha, alpha_lr, not_train, T_burnin, Csat, KI, 'upper', ahead,
        integrate=True, proportional_lr=True, scorecast=True,
    )
    etas_decaying = np.array([40 / (t ** (1 / 2 + 0.1)) for t in range(1, len(scores) + 1)])
    res_decay = decay_quantile(scores, scores[0], etas_decaying, alpha, ahead)

    sl = slice(-shift_location_1 - 100, None)
    above = {
        'us': res['q'][sl], 'aci': res_aci['q'][sl],
        'pid': res_pid_sc['q'][sl], 'decay': res_decay['q'][sl],
    }
    cov = {
        'us': sliding_window_average(res['cov'], 200)[sl],
        'aci': sliding_window_average(res_aci['cov'], 200)[sl],
        'pid': sliding_window_average(res_pid_sc['cov'], 200)[sl],
        'decay': sliding_window_average(res_decay['cov'], 200)[sl],
    }

    def rec(c):
        return (first_index_greater_than(c, 150, 800, 1 - alpha),
                first_index_greater_than(c, 950, 1200, 1 - alpha))
    us_ru, us_rd = rec(cov['us'])
    aci_ru, aci_rd = rec(cov['aci'])
    pid_ru, pid_rd = rec(cov['pid'])
    dec_ru, dec_rd = rec(cov['decay'])

    m = {
        'us_width': np.mean(above['us'][above['us'] > 0]),
        'us_abstention': np.sum(above['us'] < 0),
        'us_min_cov': np.min(cov['us']),
        'us_max_vs_aci': np.max(cov['us'] - cov['aci']),
        'us_max_vs_pid': np.max(cov['us'] - cov['pid']),
        'us_max_vs_decay': np.max(cov['us'] - cov['decay']),
        'us_arg_aci': np.argmax(cov['us'] - cov['aci']),
        'us_arg_pid': np.argmax(cov['us'] - cov['pid']),
        'us_arg_decay': np.argmax(cov['us'] - cov['decay']),
        'us_recovery_up': us_ru, 'us_recovery_down': us_rd,
        'aci_width': np.mean(above['aci'][above['aci'] < np.inf]),
        'aci_min_cov': np.min(cov['aci']),
        'aci_recovery_up': aci_ru, 'aci_recovery_down': aci_rd,
        'pid_width': np.mean(above['pid'][above['pid'] < np.inf]),
        'pid_min_cov': np.min(cov['pid']),
        'pid_recovery_up': pid_ru, 'pid_recovery_down': pid_rd,
        'decay_width': np.mean(above['decay'][above['decay'] < np.inf]),
        'decay_min_cov': np.min(cov['decay']),
        'decay_recovery_up': dec_ru, 'decay_recovery_down': dec_rd,
    }
    return m, {'above': above, 'cov': cov}

def _save_sweep_figures(param_name, value_str, plot, shift_location_1, gap, shift_location_2):
    above, cov = plot['above'], plot['cov']
    n_post = len(above['aci'])
    marks = [(100, 'gray', 'Shock (Upstream)'),
             (n_post - shift_location_2 - gap, 'slategray', 'Return'),
             (n_post - shift_location_2, 'black', 'Shock (Downstream)')]

    for arr, label, ls, c in [(above['us'], 'Split Residuals', '-', 'C0'),
                              (above['aci'], 'ACI (clipped)', '--', 'C1'),
                              (above['pid'], 'PID', '-.', 'C2'),
                              (above['decay'], 'OCID', ':', 'C3')]:
        plt.plot(arr, label=label, linestyle=ls, color=c)
    for x0, c, lab in marks:
        plt.axvline(x0, color=c, linestyle=':', linewidth=2, label=lab)
    plt.xlabel('Timestep'); plt.ylabel('Prediction Interval Width')
    plt.legend(loc='upper right', frameon=False, fontsize=8)
    plt.tight_layout()
    plt.savefig(f'synthetic_width_{param_name}_{value_str}.png', bbox_inches='tight', dpi=300)
    plt.show()

    for arr, label, ls, c in [(cov['us'], 'Split Residuals', '-', 'C0'),
                              (cov['aci'], 'ACI (clipped)', '--', 'C1'),
                              (cov['pid'], 'PID', '-.', 'C2'),
                              (cov['decay'], 'OCID', ':', 'C3')]:
        plt.plot(arr, label=label, linestyle=ls, color=c)
    plt.axhline(0.9, color='gray', linestyle='--', linewidth=1, label='90% Coverage')
    for x0, c, lab in marks:
        plt.axvline(x0, color=c, linestyle=':', linewidth=2, label=lab)
    plt.xlabel('Timestep'); plt.ylabel('Average Coverage (200 Sliding Window)')
    plt.ylim(0.6, 1.0)
    plt.legend(loc='lower right', frameon=False, fontsize=7)
    plt.savefig(f'synthetic_cov_{param_name}_{value_str}.png', bbox_inches='tight', dpi=300)
    plt.show()

def run_adaptive_sweep(param_name, config_key, param_values, base_config,
                       save_rows, numrep=50):
    """Sweep `config_key` over `param_values`; save <param_name>_adaptive.npy
    (rows = `save_rows`) and the first-rep figures for each value."""
    agg = {k: [] for k in _GAMMA_ROWS}
    for value in param_values:
        config = dict(base_config)
        config[config_key] = value
        per_value = {k: [] for k in _GAMMA_ROWS}
        for i in range(numrep):
            m, plot = _adaptive_sweep_rep(**config)
            for k in _GAMMA_ROWS:
                per_value[k].append(m[k])
            if i == 0:
                value_str = f'{value:.2f}'.replace('.', '_')
                _save_sweep_figures(param_name, value_str, plot,
                                    config['shift_location_1'], config['gap'],
                                    config['shift_location_2'])
        for k in _GAMMA_ROWS:
            arr = np.asarray(per_value[k])
            agg[k].append([np.mean(arr), np.std(arr)])
    mat = np.vstack([np.array(agg[k]).flatten() for k in save_rows])
    np.save(f'{param_name}_adaptive', mat)
    return agg

In [ ]:
# Shared config for all four sweeps (window_length=100 / cal_size=75 /
# T_burnin=100 stay fixed even in the window sweep, matching the original).
np.random.seed(42)
sweep_config = dict(
    alpha=0.1, epsilon=0.05, delta=0.3,
    window_length=100, cal_size=3 * 100 // 4, T_burnin=100,
    lr=0.01, alpha_lr=0.01, ahead=1,
    lambda_set=lambda_set, size=2100, train_size=500,
    shift_location_1=1200, gap=400, shift_location_2=400, reg=0.1,
)

### gamma sweep (`alpha_lr`) 

In [ ]:
run_adaptive_sweep('gamma', 'alpha_lr', [0.05, 0.01, 0.03, 0.05, 0.1],
                   sweep_config, save_rows=_GAMMA_ROWS)

### eta sweep (`lr`) 

In [ ]:
run_adaptive_sweep('eta', 'lr', [0.01, 0.03, 0.05, 0.1],
                   sweep_config, save_rows=_US_ROWS)

### delta sweep 

In [ ]:
run_adaptive_sweep('delta', 'delta', [0.1, 0.3, 0.5, 0.7],
                   sweep_config, save_rows=_US_ROWS)

### window sweep (`window_length`) 

In [ ]:
run_adaptive_sweep('window', 'window_length', [50, 100, 150],
                   sweep_config, save_rows=_US_ROWS)